In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"]="6"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
import torch.nn as nn
from torch import optim
import copy
import gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import math
from typing import Dict, Mapping, Optional, Tuple, Any, Union
import time
import scanpy as sc
import scipy
import scipy.sparse as sp
import anndata
from anndata import AnnData
from tqdm.notebook import tqdm
import random
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from Imputation_model.adj import intra_exp_adj, inter_adj, adj_normalize
from Imputation_model.GAE import CosMxGAE
from Imputation_model.utils import sparse_to_tuple, weighted_MSELoss, get_weight_matrix, load_preprocess
from Imputation_model.normalization import ref_query_norm, RNA_scale



"""
ref_PATH: File path to the reference (wtx) dataset, in either .h5ad or .csv format.
query_PATH: File path to the query (panel) dataset, also in .h5ad or .csv format.
save_PATH: Directory where the results will be saved.
ref_subsample_num: Number of cells to randomly sample from the reference dataset. Useful for reducing training time or avoiding GPU memory issues when the reference dataset is large.
"""
ref_PATH = "/share/fsmresfiles/UC/AtoMx/UC_1k_6k_wholetrans/whole_trans/Processed_I0261_I0331/v7/h5ad_obj/qc_wtx_v7_I0261_I0331_anno.csv"
query_PATH = "/share/fsmresfiles/UC/AtoMx/UC_1k_6k_wholetrans/6k/Processed_merged/h5ad_obj/adata_norm_wreduc_anno.csv"
save_PATH = "/share/fsmresfiles/UC/scOmics_model/Gene_imputation_model/results"
ref_subsample_num = 30000 ## [None, int] Subsample to a fraction of cells from reference data
scale_type = 'gene' ## 'cell', 'gene', False
log1p = True

device = torch.device('cuda:1') ## 'cuda:', 'cpu'
seed = 0
epochs = 10000  ## recommend >=10000
display_epochs = 500
hidden_dim = None
lr = 0.00001
dropout = 0.0


## Load data
print("Loading and preprocessing ...")
sc_RNA, sc_RNA_ref_normalized, sc_RNA_query_normalized, common_genes, pred_genes = load_preprocess(ref_PATH, query_PATH, ref_subsample_num, log1p, scale_type)



## Create training, validation, test datasets
from sklearn.model_selection import train_test_split

sc_RNA.obs['tr_test'] = ['train']*sc_RNA_ref_normalized.shape[0] + ['test']*sc_RNA_query_normalized.shape[0]
sc_RNA.obs['old_index'] = sc_RNA.obs_names
sc_RNA.obs.index = range(sc_RNA.shape[0])

train_valid_idx = sc_RNA.obs.index[:sc_RNA_ref_normalized.shape[0]]
train_idx, val_idx = train_test_split(train_valid_idx, test_size=0.2, random_state=0, shuffle=True, stratify=None)
test_idx = sc_RNA.obs.index[sc_RNA_ref_normalized.shape[0]:]

gene_to_id = {j:i for i,j in enumerate(sc_RNA.var_names.tolist())}
id_to_gene = {i:j for i,j in enumerate(sc_RNA.var_names.tolist())}

seq_RNA = [gene_to_id[k] for k in sc_RNA.var_names]
seq_RNA = torch.LongTensor(seq_RNA)

print("Training (ref) cells:", len(train_idx), "\tValidation (ref) cells:", len(val_idx), "\tTest (query) cells:", len(test_idx))



## Compute adjacency matrix
from scipy.sparse import block_diag, vstack, hstack, identity

print("Computing adjacency matrix ...")
sc_RNA_input = sc_RNA[:, common_genes]

sc.tl.pca(sc_RNA_input, n_comps=30, svd_solver='arpack', random_state=0)

sc_RNA_input.obsp['inter_adj'] = inter_adj(sc_RNA_input[train_valid_idx].obsm['X_pca'], sc_RNA_input[test_idx].obsm['X_pca'], find_neighbor_method='MNN', dist_method='euclidean', corr_dist_neighbors=20)

A_train = intra_exp_adj(sc_RNA_input[train_valid_idx].obsm['X_pca'], find_neighbor_method='MNN', dist_method='euclidean', corr_dist_neighbors=20)
A_test = intra_exp_adj(sc_RNA_input[test_idx].obsm['X_pca'], find_neighbor_method='MNN', dist_method='euclidean', corr_dist_neighbors=20)

sc_RNA_input.obsp['intra_adj_tr_test'] = block_diag((A_train, A_test), format="csr")

try:
    adj_exp = sc_RNA_input.obsp['inter_adj'] + sc_RNA_input.obsp['intra_adj_train'] + sc_RNA_input.obsp['intra_adj_test'] + sp.identity(sc_RNA_input.shape[0], format='csr')
except:
    adj_exp = sc_RNA_input.obsp['inter_adj'] + sc_RNA_input.obsp['intra_adj_tr_test'] + sp.identity(sc_RNA_input.shape[0], format='csr')
sc_RNA_input.obsp['MNN_connectivities_norm'] = adj_normalize(adj_exp, symmetry=True)

print("Input genes:", sc_RNA_input.shape[1], "Output genes:", sc_RNA.shape[1])



## Adjust model input format
print("Completing input format ...")
input_features = sc_RNA_input.X
n_nodes, input_dim = input_features.shape

input_features = sparse_to_tuple(scipy.sparse.csr_matrix(input_features))
input_features = torch.sparse.FloatTensor(torch.LongTensor(input_features[0].T), torch.FloatTensor(input_features[1]), torch.Size(input_features[2])).to(device)

adj_RNA = sc_RNA_input.obsp['MNN_connectivities_norm']
adj_RNA = sparse_to_tuple(adj_RNA)
adj_RNA = torch.sparse.FloatTensor(torch.LongTensor(adj_RNA[0].T), torch.FloatTensor(adj_RNA[1]), torch.Size(adj_RNA[2])).to(device)

output_features = torch.tensor(sc_RNA.X.A).to(device)

weight_tensor_output = get_weight_matrix(np.array(output_features[:sc_RNA_ref_normalized.shape[0]].cpu()), power=1/6)
weight_tensor_output = torch.Tensor(np.vstack([weight_tensor_output, np.ones([sc_RNA_query_normalized.shape[0], weight_tensor_output.shape[1]])]))
sc_RNA.layers['training_weights'] = np.array(weight_tensor_output)



## Model training
print("Start model training ...")
if hidden_dim == None:
    hidden_dim = input_features.shape[1]

model = CosMxGAE(adj_RNA, input_features.shape[1], output_features.shape[1], hidden_dim, dropout).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=200)

predict_criterion = weighted_MSELoss(reduction='mean')

best_val_loss = np.inf

t = time.time()
running_loss = []
for epoch in tqdm(range(epochs)):
    
    model.train()
    optimizer.zero_grad()
    
    pred_features = model(input_features.float())
    
    loss = predict_criterion(pred_features[train_idx], output_features[train_idx], weights=weight_tensor_output[train_idx].to(device))

    loss.backward()
    optimizer.step()
    
    del pred_features
    gc.collect()
    torch.cuda.empty_cache()
    
    model.eval()
    with torch.no_grad():
        
        pred_features = model(input_features.float())
        
        val_loss = predict_criterion(pred_features[val_idx], output_features[val_idx], weights=weight_tensor_output[val_idx].to(device))
        test_loss = predict_criterion(pred_features[len(train_valid_idx):], output_features[len(train_valid_idx):], weights=weight_tensor_output[len(train_valid_idx):].to(device))
        
        running_loss.append([loss.item(), val_loss.item(), test_loss.item()])
    
        if val_loss < best_val_loss:
            best_epoch = epoch
            best_val_loss = val_loss
            best_test_loss = test_loss
            best_pred = pred_features.detach().cpu().clone()
            best_model_wts = copy.deepcopy(model.state_dict())

        if (epoch + 1) % display_epochs == 0:
            print("Epoch:", '%04d' % (epoch + 1), "train_loss=", "{:.5f}".format(loss.item()), "val_loss=", "{:.5f}".format(val_loss.item()), "test_loss=", "{:.5f}".format(test_loss.item()), "time=", "{:.5f} mins".format((time.time() - t)/60.0))

        scheduler.step(val_loss)
        
        if epoch+1 < epochs:
            del pred_features, loss, val_loss, test_loss
            gc.collect()
            torch.cuda.empty_cache()

print("Optimization Finished!")



print(f"Saving model parameters to {save_PATH}/model_best_wts.pt ...")
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), f'{save_PATH}/model_best_wts.pt')
## Load
#model = CosMxGAE(...)  ## Re-initialize your model with the same architecture
#model.load_state_dict(torch.load(f'{save_PATH}/model_best_wts.pt'))
#model.eval()


print(f"Saving results to {save_PATH} ...")
if scale_type == 'gene':
    sc_RNA.X = sp.csr_matrix(sc_RNA.X.A * sc_RNA.var['gene_max_ref'].values)
    sc_RNA.layers['RNA_pred_f_log1p'] = np.clip(np.array(pred_features.detach().cpu().clone()) * sc_RNA.var['gene_max_ref'].values, 0, np.inf)
    sc_RNA.layers['RNA_pred_b_log1p'] = np.clip(np.array(best_pred) * sc_RNA.var['gene_max_ref'].values, 0, np.inf)
elif scale_type == 'cell':
    sc_RNA.X = sp.csr_matrix((sc_RNA.X.A.T * sc_RNA.obs['cell_max_common_genes'].values).T)
    sc_RNA.layers['RNA_pred_f_log1p'] = np.clip(np.array(pred_features.detach().cpu().clone().T * sc_RNA.obs['cell_max_common_genes'].values).T, 0, np.inf)
    sc_RNA.layers['RNA_pred_b_log1p'] = np.clip(np.array(best_pred.T * sc_RNA.obs['cell_max_common_genes'].values).T, 0, np.inf)
else:
    sc_RNA.layers['RNA_pred_f_log1p'] = np.clip(np.array(pred_features.detach().cpu().clone()), 0, np.inf)
    sc_RNA.layers['RNA_pred_b_log1p'] = np.clip(np.array(best_pred), 0, np.inf)

sc_RNA_test = sc_RNA[test_idx]
#sc_RNA_test.layers['RNA_pred_f'] = np.expm1(sc_RNA_test.layers['RNA_pred_f_log1p'])
#sc_RNA_test.layers['RNA_pred_b'] = np.expm1(sc_RNA_test.layers['RNA_pred_b_log1p'])

sc_RNA_test.obs_names = sc_RNA_test.obs['old_index'].values
sc_RNA_test.obs.index.name = None
sc_RNA_test_unique_gene = sc_RNA_test[:, pred_genes]

pd.DataFrame(sc_RNA_test_unique_gene.layers['RNA_pred_f_log1p'], index = sc_RNA_test_unique_gene.obs['old_index'], columns = sc_RNA_test_unique_gene.var_names).to_csv(f"{save_PATH}/Gene_imputation_final_log1p.csv", index=True, header=True)
pd.DataFrame(sc_RNA_test_unique_gene.layers['RNA_pred_b_log1p'], index = sc_RNA_test_unique_gene.obs['old_index'], columns = sc_RNA_test_unique_gene.var_names).to_csv(f"{save_PATH}/Gene_imputation_best_log1p.csv", index=True, header=True)
#pd.DataFrame(sc_RNA_test_unique_gene.layers['RNA_pred_f'], index = sc_RNA_test_unique_gene.obs['old_index'], columns = sc_RNA_test_unique_gene.var_names).to_csv(f"{save_PATH}/Gene_imputation_final.csv", index=True, header=True)
#pd.DataFrame(sc_RNA_test_unique_gene.layers['RNA_pred_b'], index = sc_RNA_test_unique_gene.obs['old_index'], columns = sc_RNA_test_unique_gene.var_names).to_csv(f"{save_PATH}/Gene_imputation_best.csv", index=True, header=True)

print("Predicted gene num:", sc_RNA_test_unique_gene.shape[1])
print(f"Imputation results have been saved to {save_PATH}")

Loading and preprocessing ...
Training (ref) cells: 24000 	Validation (ref) cells: 6000 	Test (query) cells: 57654
Computing adjacency matrix ...
Input genes: 6115 Output genes: 18935
Completing input format ...
Weight of non-zero values: 1.5463000380552718
Start model training ...


  0%|          | 0/600 [00:00<?, ?it/s]

Epoch: 0500 train_loss= 0.01035 val_loss= 0.01056 test_loss= 0.00759 time= 66.05963 mins
Optimization Finished!
Saving model parameters to /share/fsmresfiles/UC/scOmics_model/Gene_imputation_model/results/model_best_wts.pt ...
Saving results to /share/fsmresfiles/UC/scOmics_model/Gene_imputation_model/results ...
Predicted gene num: 12820
Imputation results have been saved to /share/fsmresfiles/UC/scOmics_model/Gene_imputation_model/results
